# LAID: Kaggle GPU bootstrap

Use this notebook to verify the Kaggle runtime and bootstrap the public benchmark repository without stressing your local computer.

Before **Run All**, open **Settings** in the right sidebar, choose a GPU accelerator, and turn **Internet on**. If Kaggle asks for phone verification, complete that first.

In [ ]:
import shutil
import subprocess
from pathlib import Path

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is missing from this runtime.') from exc

if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Notebook options, select a GPU accelerator, save, and reconnect.')

gpu = torch.cuda.get_device_properties(0)
disk = shutil.disk_usage('/kaggle/working')
print(f'GPU: {gpu.name}')
print(f'GPU memory: {gpu.total_memory / 1024**3:.1f} GB')
print(f'Free working disk: {disk.free / 1024**3:.1f} GB')


In [ ]:
repo = Path('/kaggle/working/laid')
if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not a Git checkout. Remove or rename it, then rerun.')
else:
    subprocess.run(['git', 'clone', 'https://github.com/Eienel/laid.git', str(repo)], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-e', str(repo)], check=True)


In [ ]:
subprocess.run([
    'python', '-m', 'bakeoff.cli', 'preflight',
    '--path', str(repo), '--require-safe',
    '--min-ram-gb', '4', '--min-disk-gb', '10'
], check=True)
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', str(repo / 'tests'), '-v'], check=True)


## Ready

If all three code cells pass, the GPU runtime and LAID benchmark harness are ready. Stop the Kaggle session when you finish so its quota does not keep running. The next notebook revision will download and benchmark the first detector candidate.